In [ ]:
RUN_BASELINE = True     # BERT supervised baseline (~8 min)
RUN_COT      = False    # set True later for the chain-of-thought check (~10 min)
BS           = 32       # lower to 16 if you get an out-of-memory error
COT_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct"
COT_N        = 200

import glob, os, pickle, xml.etree.ElementTree as ET
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from scipy.special import log_softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForMaskedLM, AutoModelForCausalLM)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU! Runtime -> Change runtime type -> T4 GPU, then Connect and re-run."
DT = torch.float16
LABELS = ["positive", "negative", "neutral"]
def score(y, p): return accuracy_score(y, p), f1_score(y, p, average="macro")

# ---------------------------------------------------------------- 1. DATA
root = snapshot_download("alexcadillon/SemEval2014Task4", repo_type="dataset", allow_patterns=["*.xml"])
xmls = sorted(glob.glob(f"{root}/**/*.xml", recursive=True))
def pick(*keys):
    c = [p for p in xmls if all(k in p.lower() for k in keys)]
    c = sorted(c, key=lambda p: "v2" not in p.lower())
    assert c, f"no file for {keys}: {[os.path.basename(p) for p in xmls]}"
    return c[0]

def load(path):
    rows = []
    for s in ET.parse(path).getroot().iter("sentence"):
        ats = s.find("aspectTerms")
        if ats is None: continue
        text = s.find("text").text
        for a in ats.findall("aspectTerm"):
            if a.get("polarity") in LABELS:                       # drops 'conflict'
                rows.append((text, a.get("term"), LABELS.index(a.get("polarity"))))
    return rows

D = {"R_tr": load(pick("restaurant", "train")), "R_te": load(pick("restaurant", "test_gold")),
     "L_tr": load(pick("laptop", "train")),     "L_te": load(pick("laptop", "test_gold"))}
for k, v in D.items():
    print(k, len(v), "pos/neg/neu =", np.bincount([r[2] for r in v], minlength=3))

# 2. PROMPT FEATURES (cached)
CACHE = "/content/feats.pkl"
if os.path.exists(CACHE):
    F = pickle.load(open(CACHE, "rb")); print("Loaded cached features.")
else:
    # NLI branch: review = premise, "The {aspect} is good/bad/okay." = hypothesis
    ntok = AutoTokenizer.from_pretrained("roberta-large-mnli")
    nli = AutoModelForSequenceClassification.from_pretrained("roberta-large-mnli", torch_dtype=DT).to(dev).eval()
    ENT = nli.config.label2id["ENTAILMENT"]
    HYP = ["The {} is good.", "The {} is bad.", "The {} is okay."]

    @torch.no_grad()
    def nli_feats(rows):
        out = []
        for i in tqdm(range(0, len(rows), BS), desc="NLI branch"):
            b, f = rows[i:i+BS], []
            for h in HYP:
                e = ntok([r[0] for r in b], [h.format(r[1]) for r in b], return_tensors="pt",
                         padding=True, truncation=True, max_length=256).to(dev)
                f.append(nli(**e).logits[:, ENT].float().cpu())
            out.append(torch.stack(f, 1))
        return log_softmax(torch.cat(out).numpy(), axis=1)

    # Cloze branch: "<review> The {aspect} is <mask>."
    mtok = AutoTokenizer.from_pretrained("roberta-large")
    mlm = AutoModelForMaskedLM.from_pretrained("roberta-large", torch_dtype=DT).to(dev).eval()
    WORDS = [[" good", " great"], [" bad", " terrible"], [" okay", " fine"]]
    IDS = [[mtok.encode(w, add_special_tokens=False)[0] for w in ws] for ws in WORDS]

    @torch.no_grad()
    def cloze_feats(rows):
        out = []
        for i in tqdm(range(0, len(rows), BS), desc="Cloze branch"):
            b = rows[i:i+BS]
            e = mtok([f"{r[0]} The {r[1]} is <mask>." for r in b], return_tensors="pt", padding=True).to(dev)
            pos = (e.input_ids == mtok.mask_token_id).nonzero()
            lp = mlm(**e).logits[pos[:, 0], pos[:, 1]].float().log_softmax(-1)
            out.append(torch.stack([lp[:, g].logsumexp(-1) for g in IDS], 1).cpu())
        return log_softmax(torch.cat(out).numpy(), axis=1)

    F = {k: dict(nli=nli_feats(v), cloze=cloze_feats(v), y=np.array([r[2] for r in v])) for k, v in D.items()}
    pickle.dump(F, open(CACHE, "wb"))
    del nli, mlm; torch.cuda.empty_cache()

# 3. EVALUATION
def X(d, mode):
    return {"nli": d["nli"], "cloze": d["cloze"], "both": np.hstack([d["nli"], d["cloze"]])}[mode]

results = []
for te in ["R_te", "L_te"]:                                          # zero-shot
    d = F[te]
    for name, s in [("prompt-nli", d["nli"]), ("prompt-cloze", d["cloze"]), ("prompt-both", d["nli"] + d["cloze"])]:
        a, f = score(d["y"], s.argmax(1))
        results.append(dict(method=name, setting=f"{te[0]} (zero-shot)", n=0, acc=a, f1=f))

SETTINGS = [("R_tr", "R_te", "R->R"), ("L_tr", "L_te", "L->L"), ("R_tr", "L_te", "R->L"), ("L_tr", "R_te", "L->R")]
for mode in ["nli", "cloze", "both"]:                                # few-shot, full, cross-domain
    for tr, te, tag in SETTINGS:
        Xtr, ytr, Xte, yte = X(F[tr], mode), F[tr]["y"], X(F[te], mode), F[te]["y"]
        for n in [16, 64, 256, 1024, "full"]:
            m = len(ytr) if n == "full" else min(n, len(ytr))
            r = []
            for s in range(1 if n == "full" else 5):
                idx = np.random.RandomState(s).permutation(len(ytr))[:m]
                clf = LogisticRegression(max_iter=3000).fit(Xtr[idx], ytr[idx])
                r.append(score(yte, clf.predict(Xte)))
            a, f = np.mean(r, 0)
            results.append(dict(method=f"prompt-{mode}", setting=tag, n=m, acc=a, f1=f))
print("Prompt-based evaluation done.")

# 4. SUPERVISED BERT BASELINE
if RUN_BASELINE:
    btok = AutoTokenizer.from_pretrained("bert-base-uncased")
    def enc(rows):
        return btok([r[0] for r in rows], [r[1] for r in rows], truncation=True, max_length=128,
                    padding=True, return_tensors="pt")
    def run_bert(train_rows, test_sets, epochs, seed=0, bs=32, lr=2e-5):
        torch.manual_seed(seed)
        model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=lr)
        rng = np.random.RandomState(seed)
        for _ in range(epochs):
            model.train()
            order = rng.permutation(len(train_rows))
            for i in range(0, len(order), bs):
                b = [train_rows[j] for j in order[i:i+bs]]
                e = {k: v.to(dev) for k, v in enc(b).items()}
                loss = model(**e, labels=torch.tensor([r[2] for r in b]).to(dev)).loss
                loss.backward(); opt.step(); opt.zero_grad()
        model.eval(); out = {}
        for name, rows in test_sets.items():
            preds = []
            with torch.no_grad():
                for i in range(0, len(rows), 64):
                    e = {k: v.to(dev) for k, v in enc(rows[i:i+64]).items()}
                    preds += model(**e).logits.argmax(-1).cpu().tolist()
            out[name] = score(np.array([r[2] for r in rows]), np.array(preds))
        return out
    for trk, t_in, t_x in [("R_tr", "R_te", "L_te"), ("L_tr", "L_te", "R_te")]:
        for n in [256, "full"]:
            rows = D[trk] if n == "full" else [D[trk][i] for i in np.random.RandomState(0).permutation(len(D[trk]))[:n]]
            print(f"BERT baseline: train {trk}, n={len(rows)} ...")
            res = run_bert(rows, {t_in: D[t_in], t_x: D[t_x]}, epochs=3 if n == "full" else 10)
            for te, (a, f) in res.items():
                results.append(dict(method="bert-head", setting=f"{trk[0]}->{te[0]}", n=len(rows), acc=a, f1=f))


R = pd.DataFrame(results); R.to_csv("atsc_results.csv", index=False)
pd.set_option("display.width", 200)
print("\n=== ZERO-SHOT ===");  print(R[R.setting.str.contains("zero")].round(3).to_string(index=False))
print("\n=== MACRO-F1 (rows: method, n | columns: train->test domain) ===")
print(R[~R.setting.str.contains("zero")].pivot_table(index=["method", "n"], columns="setting", values="f1").round(3))

# 5.CHAIN-OF-THOUGHT CHECK
if RUN_COT:
    gtok = AutoTokenizer.from_pretrained(COT_MODEL)
    gen = AutoModelForCausalLM.from_pretrained(COT_MODEL, torch_dtype=DT).to(dev).eval()
    DIRECT = ("Review: {t}\nAspect: {a}\nWhat is the sentiment toward this aspect? "
              "Answer with one word: positive, negative or neutral.")
    COT = ("Review: {t}\nAspect: {a}\nStep 1: Which words or phrases in the review express an opinion about "
           "this aspect?\nStep 2: Is that opinion positive, negative or neutral?\n"
           "End with a final line 'Answer: <positive|negative|neutral>'.")
    @torch.no_grad()
    def ask(tmpl, t, a, max_new):
        msgs = [{"role": "user", "content": tmpl.format(t=t, a=a)}]
        ids = gtok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(dev)
        out = gen.generate(**ids, max_new_tokens=max_new, do_sample=False)
        return gtok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).lower()
    def parse(txt):
        txt = txt.split("answer:")[-1]
        hits = [(txt.rfind(l), i) for i, l in enumerate(LABELS) if l in txt]
        return max(hits)[1] if hits else 2
    sub = [D["R_te"][i] for i in np.random.RandomState(1).permutation(len(D["R_te"]))[:COT_N]]
    y = np.array([r[2] for r in sub])
    p_dir = np.array([parse(ask(DIRECT, r[0], r[1], 6)) for r in tqdm(sub, desc="direct prompt")])
    cot_out = [ask(COT, r[0], r[1], 120) for r in tqdm(sub, desc="CoT prompt")]
    p_cot = np.array([parse(o) for o in cot_out])
    print(f"\n=== LLM PROMPTING ({COT_MODEL}, {len(sub)} examples) ===")
    print("direct acc/F1:", np.round(score(y, p_dir), 3), "| CoT acc/F1:", np.round(score(y, p_cot), 3))
    flips = [i for i in range(len(sub)) if p_dir[i] != p_cot[i]]
    print(f"{len(flips)} predictions changed by reasoning: "
          f"{sum(p_cot[i] == y[i] for i in flips)} fixed, {sum(p_dir[i] == y[i] for i in flips)} broken")
    with open("cot_flips.txt", "w") as fh:
        for i in flips:
            fh.write(f"REVIEW: {sub[i][0]}\nASPECT: {sub[i][1]} | gold={LABELS[y[i]]} direct={LABELS[p_dir[i]]} "
                     f"cot={LABELS[p_cot[i]]}\nREASONING: {cot_out[i]}\n{'-'*60}\n")


try:
    from google.colab import files
    files.download("atsc_results.csv")
    if RUN_COT: files.download("cot_flips.txt")
except Exception as e:
    print("Results are saved as atsc_results.csv in the Colab file panel (left sidebar).", e)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

R_tr 3602 pos/neg/neu = [2164  805  633]
R_te 1120 pos/neg/neu = [728 196 196]
L_tr 2313 pos/neg/neu = [987 866 460]
L_te 638 pos/neg/neu = [341 128 169]


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== ZERO-SHOT ===
      method       setting  n   acc    f1
  prompt-nli R (zero-shot)  0 0.836 0.741
prompt-cloze R (zero-shot)  0 0.748 0.548
 prompt-both R (zero-shot)  0 0.792 0.619
  prompt-nli L (zero-shot)  0 0.749 0.691
prompt-cloze L (zero-shot)  0 0.683 0.583
 prompt-both L (zero-shot)  0 0.727 0.633

=== MACRO-F1 (rows: method, n | columns: train->test domain) ===
setting             L->L   L->R   R->L   R->R
method       n                               
bert-head    256   0.610  0.525  0.511  0.590
             2313  0.701  0.656    NaN    NaN
             3602    NaN    NaN  0.702  0.752
prompt-both  16    0.657  0.662  0.583  0.588
             64    0.720  0.706  0.685  0.694
             256   0.737  0.722  0.747  0.730
             1024  0.738  0.720  0.750  0.726
             2313  0.732  0.720    NaN    NaN
             3602    NaN    NaN  0.741  0.723
prompt-cloze 16    0.547  0.567  0.561  0.575
             64    0.569  0.580  0.567  0.583
             256   0.55

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============ OBJECTIVE 2: implicit aspect categories + chain-of-thought ============
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PER_GROUP = 100          # 100 category-word-absent + 100 present = 200 LLM examples
BS = 32

import glob, os, pickle, xml.etree.ElementTree as ET
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from scipy.special import log_softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForMaskedLM, AutoModelForCausalLM)
dev, DT = "cuda", torch.float16
assert torch.cuda.is_available(), "Turn on the T4 GPU runtime first."
LABELS = ["positive", "negative", "neutral"]
def score(y, p): return round(accuracy_score(y, p), 3), round(f1_score(y, p, average="macro"), 3)

# ---- data: restaurant train (aspect terms) and restaurant test (aspect CATEGORIES)
root = snapshot_download("alexcadillon/SemEval2014Task4", repo_type="dataset", allow_patterns=["*.xml"])
xmls = sorted(glob.glob(f"{root}/**/*.xml", recursive=True))
def pick(*keys):
    c = [p for p in xmls if all(k in p.lower() for k in keys)]
    return sorted(c, key=lambda p: "v2" not in p.lower())[0]
def load_terms(path):
    rows = []
    for s in ET.parse(path).getroot().iter("sentence"):
        ats = s.find("aspectTerms")
        if ats is None: continue
        for a in ats.findall("aspectTerm"):
            if a.get("polarity") in LABELS:
                rows.append((s.find("text").text, a.get("term"), LABELS.index(a.get("polarity"))))
    return rows
def load_cats(path):
    rows = []
    for s in ET.parse(path).getroot().iter("sentence"):
        cs = s.find("aspectCategories")
        if cs is None: continue
        for c in cs.findall("aspectCategory"):
            if c.get("polarity") in LABELS and c.get("category") != "anecdotes/miscellaneous":
                rows.append((s.find("text").text, c.get("category"), LABELS.index(c.get("polarity"))))
    return rows
R_tr = load_terms(pick("restaurant", "train"))
C_te = load_cats(pick("restaurant", "test_gold"))
yc = np.array([r[2] for r in C_te])
absent = np.array([r[1].lower() not in r[0].lower() for r in C_te])   # category word not in text = implicit proxy
print("category queries:", len(C_te), "| word absent from text (implicit proxy):", int(absent.sum()),
      "| present:", int((~absent).sum()))

# ---- prompt features (NLI + cloze)
ntok = AutoTokenizer.from_pretrained("roberta-large-mnli")
nli = AutoModelForSequenceClassification.from_pretrained("roberta-large-mnli", torch_dtype=DT).to(dev).eval()
ENT = nli.config.label2id["ENTAILMENT"]
HYP = ["The {} is good.", "The {} is bad.", "The {} is okay."]
@torch.no_grad()
def nli_feats(rows):
    out = []
    for i in tqdm(range(0, len(rows), BS), desc="NLI"):
        b, f = rows[i:i+BS], []
        for h in HYP:
            e = ntok([r[0] for r in b], [h.format(r[1]) for r in b], return_tensors="pt",
                     padding=True, truncation=True, max_length=256).to(dev)
            f.append(nli(**e).logits[:, ENT].float().cpu())
        out.append(torch.stack(f, 1))
    return log_softmax(torch.cat(out).numpy(), axis=1)
mtok = AutoTokenizer.from_pretrained("roberta-large")
mlm = AutoModelForMaskedLM.from_pretrained("roberta-large", torch_dtype=DT).to(dev).eval()
WORDS = [[" good", " great"], [" bad", " terrible"], [" okay", " fine"]]
IDS = [[mtok.encode(w, add_special_tokens=False)[0] for w in ws] for ws in WORDS]
@torch.no_grad()
def cloze_feats(rows):
    out = []
    for i in tqdm(range(0, len(rows), BS), desc="Cloze"):
        b = rows[i:i+BS]
        e = mtok([f"{r[0]} The {r[1]} is <mask>." for r in b], return_tensors="pt", padding=True).to(dev)
        pos = (e.input_ids == mtok.mask_token_id).nonzero()
        lp = mlm(**e).logits[pos[:, 0], pos[:, 1]].float().log_softmax(-1)
        out.append(torch.stack([lp[:, g].logsumexp(-1) for g in IDS], 1).cpu())
    return log_softmax(torch.cat(out).numpy(), axis=1)

CACHE = "/content/feats.pkl"
Ftr = pickle.load(open(CACHE, "rb"))["R_tr"] if os.path.exists(CACHE) else \
      dict(nli=nli_feats(R_tr), cloze=cloze_feats(R_tr), y=np.array([r[2] for r in R_tr]))
Fc = dict(nli=nli_feats(C_te), cloze=cloze_feats(C_te))
del nli, mlm; torch.cuda.empty_cache()

rows_out = []
def report(name, pred, mask=None):
    m = np.ones(len(yc), bool) if mask is None else mask
    for grp, gm in [("all", m), ("word absent (implicit)", m & absent), ("word present", m & ~absent)]:
        if gm.sum(): rows_out.append(dict(method=name, group=grp, n=int(gm.sum()),
                                          acc=score(yc[gm], pred[gm])[0], f1=score(yc[gm], pred[gm])[1]))

# ---- A) zero-shot NLI on category queries; B) trained on aspect TERMS, tested on CATEGORIES (ATSC->ACSC)
report("NLI zero-shot", Fc["nli"].argmax(1))
for mode in ["nli", "both"]:
    Xtr = Ftr["nli"] if mode == "nli" else np.hstack([Ftr["nli"], Ftr["cloze"]])
    Xte = Fc["nli"] if mode == "nli" else np.hstack([Fc["nli"], Fc["cloze"]])
    clf = LogisticRegression(max_iter=3000).fit(Xtr, Ftr["y"])
    report(f"ATSC->ACSC ({mode}, terms-trained)", clf.predict(Xte))

# ---- C) LLM: direct prompt vs chain-of-thought, on a balanced subset
rng = np.random.RandomState(1)
sel = np.concatenate([rng.choice(np.where(absent)[0], min(PER_GROUP, int(absent.sum())), replace=False),
                      rng.choice(np.where(~absent)[0], PER_GROUP, replace=False)])
sub = [C_te[i] for i in sel]; ys = yc[sel]; ab = absent[sel]

gtok = AutoTokenizer.from_pretrained(LLM_NAME)
gen = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=DT).to(dev).eval()
DIRECT = ("Review: {t}\nAspect category: {a}\nWhat is the sentiment of the review toward this aspect category? "
          "Answer with one word: positive, negative or neutral.")
COT = ("Review: {t}\nAspect category: {a}\nStep 1: Which words or phrases in the review express an opinion "
       "related to this aspect category, even if the category is not named?\nStep 2: Is that opinion positive, "
       "negative or neutral?\nEnd with a final line 'Answer: <positive|negative|neutral>'.")
@torch.no_grad()
def ask(tmpl, t, a, max_new):
    ids = gtok.apply_chat_template([{"role": "user", "content": tmpl.format(t=t, a=a)}],
                                   add_generation_prompt=True, return_tensors="pt", return_dict=True).to(dev)
    out = gen.generate(**ids, max_new_tokens=max_new, do_sample=False)
    return gtok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).lower()
def parse(txt):
    txt = txt.split("answer:")[-1]
    hits = [(txt.rfind(l), i) for i, l in enumerate(LABELS) if l in txt]
    return max(hits)[1] if hits else 2
p_dir = np.array([parse(ask(DIRECT, r[0], r[1], 6)) for r in tqdm(sub, desc="LLM direct")])
cot_out = [ask(COT, r[0], r[1], 120) for r in tqdm(sub, desc="LLM CoT")]
p_cot = np.array([parse(o) for o in cot_out])
nli_sub = Fc["nli"][sel].argmax(1)
for name, p in [("NLI zero-shot (same 200)", nli_sub), ("LLM direct", p_dir), ("LLM chain-of-thought", p_cot)]:
    for grp, gm in [("all", np.ones(len(ys), bool)), ("word absent (implicit)", ab), ("word present", ~ab)]:
        rows_out.append(dict(method=name, group=grp, n=int(gm.sum()),
                             acc=score(ys[gm], p[gm])[0], f1=score(ys[gm], p[gm])[1]))

R2 = pd.DataFrame(rows_out); R2.to_csv("objective2_results.csv", index=False)
pd.set_option("display.width", 200)
print("\n===== OBJECTIVE 2 RESULTS ====="); print(R2.to_string(index=False))
flips = [i for i in range(len(sub)) if p_dir[i] != p_cot[i]]
print(f"\nReasoning changed {len(flips)} of {len(sub)} predictions: "
      f"{sum(p_cot[i]==ys[i] for i in flips)} fixed, {sum(p_dir[i]==ys[i] for i in flips)} broken")
with open("cot_flips.txt", "w") as fh:
    for i in flips:
        fh.write(f"REVIEW: {sub[i][0]}\nCATEGORY: {sub[i][1]} | gold={LABELS[ys[i]]} direct={LABELS[p_dir[i]]} "
                 f"cot={LABELS[p_cot[i]]} | category word in text: {not ab[i]}\nREASONING: {cot_out[i]}\n{'-'*60}\n")
from google.colab import files
files.download("objective2_results.csv"); files.download("cot_flips.txt")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

category queries: 754 | word absent from text (implicit proxy): 491 | present: 263


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


===== OBJECTIVE 2 RESULTS =====
                          method                  group   n   acc    f1
                   NLI zero-shot                    all 754 0.869 0.744
                   NLI zero-shot word absent (implicit) 491 0.847 0.753
                   NLI zero-shot           word present 263 0.909 0.669
 ATSC->ACSC (nli, terms-trained)                    all 754 0.893 0.721
 ATSC->ACSC (nli, terms-trained) word absent (implicit) 491 0.866 0.718
 ATSC->ACSC (nli, terms-trained)           word present 263 0.943 0.680
ATSC->ACSC (both, terms-trained)                    all 754 0.891 0.698
ATSC->ACSC (both, terms-trained) word absent (implicit) 491 0.864 0.698
ATSC->ACSC (both, terms-trained)           word present 263 0.943 0.632
        NLI zero-shot (same 200)                    all 200 0.875 0.743
        NLI zero-shot (same 200) word absent (implicit) 100 0.840 0.753
        NLI zero-shot (same 200)           word present 100 0.910 0.622
                      LLM direc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>